In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/TUM/projects/Deepfakes/Experiment_2/Phase2_combined_with_group.csv')

In [ ]:
list(df)

['participant_id',
 'age_group',
 'gender',
 'education',
 'occupation',
 'country_of_residence',
 'nationality',
 'race',
 'native_tongue',
 'languages_spoken',
 'english_fluency',
 'political_party',
 'political_inclination',
 'listening_habits',
 'tech_savy',
 'ai_experience',
 'media_consumption',
 'participant_date_of_entry',
 'rating_id',
 'audio_clip_id',
 'speech_clarity',
 'speech_persuasiveness',
 'speech_pace_engagement',
 'speaker_trustworthiness',
 'speech_trustworthiness',
 'speaker_competence',
 'speech_speed_influence',
 'pitch_sincerity_effect',
 'loudness_attention_influence',
 'realness_scale',
 'realness_perception',
 'influenced_by_tone',
 'influenced_by_quality',
 'influenced_by_content',
 'confidence_level',
 'policy_agreement',
 'likelihood_to_vote',
 'open_ended_response',
 'group_no',
 'share_likely_private',
 'share_likely_public',
 'report_misleading',
 'downrank_agree',
 'watermark_action',
 'candidate_position_after',
 'agreement_candidate_position',
 'can

#Issue salience

In [ ]:
df = df[df['political_party'].isin(['Republicans', 'Democrats'])]
df['condition_full'] = df['group'].map({1: 'Real', 2: 'fake_aligned', 3: 'fake_contradicting'})
df['stance_change'] = df['stance_after'] - df['stance_before']
df['salience_change'] = df['salience_after'] - df['salience_before']
df['party'] = df['political_party']
df_harris = df[df['speaker']=='Harris']
df_trump = df[df['speaker']=='Trump']


In [ ]:
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Ensure the data is loaded and preprocessed (Section 1) before running this section.

# Drop any participants with missing stance or salience values (to have equal N for pre and post)
df_stance = df.dropna(subset=['stance_before', 'stance_after'])
df_salience = df.dropna(subset=['salience_before', 'salience_after'])

# 2.a. Paired t-tests for overall pre vs post change in stance and salience
t_stance, p_stance = stats.ttest_rel(df_stance['stance_after'], df_stance['stance_before'])
t_salience, p_salience = stats.ttest_rel(df_salience['salience_after'], df_salience['salience_before'])
print(f"Paired t-test for stance (post vs pre): t = {t_stance:.3f}, p = {p_stance:.3f}")
print(f"Paired t-test for salience (post vs pre): t = {t_salience:.3f}, p = {p_salience:.3f}")

# 2.b. Between-group ANOVAs on change scores
# Model stance_change ~ condition_full * party
model_stance = smf.ols('stance_change ~ C(condition_full) * C(party)', data=df_stance).fit()
anova_stance = sm.stats.anova_lm(model_stance, typ=2)
print("\nANOVA: Issue Stance Change ~ Condition * Party")
print(anova_stance)

# Model salience_change ~ condition_full * party
model_salience = smf.ols('salience_change ~ C(condition_full) * C(party)', data=df_salience).fit()
anova_salience = sm.stats.anova_lm(model_salience, typ=2)
print("\nANOVA: Issue Salience Change ~ Condition * Party")
print(anova_salience)

Paired t-test for stance (post vs pre): t = 0.440, p = 0.660
Paired t-test for salience (post vs pre): t = -0.208, p = 0.836

ANOVA: Issue Stance Change ~ Condition * Party
                                 sum_sq     df         F    PR(>F)
C(condition_full)             20.759963    2.0  2.386042  0.093114
C(party)                       1.457162    1.0  0.334957  0.563034
C(condition_full):C(party)    11.207250    2.0  1.288103  0.276772
Residual                    2031.586818  467.0       NaN       NaN

ANOVA: Issue Salience Change ~ Condition * Party
                                sum_sq     df         F    PR(>F)
C(condition_full)             8.729497    2.0  2.473307  0.085409
C(party)                      0.002751    1.0  0.001559  0.968525
C(condition_full):C(party)    1.058443    2.0  0.299886  0.741045
Residual                    824.134485  467.0       NaN       NaN


In [ ]:
model_private = smf.ols('share_likely_private ~ C(condition_full)', data=df).fit()
print("ALL")
print(model_private.summary2())

model_private = smf.ols('share_likely_private ~ C(condition_full) * C(party, )', data=df_harris).fit()
print("Harris")

print(model_private.summary2())

model_private = smf.ols('share_likely_private ~ C(condition_full) * C(party, Treatment(reference="Republicans"))', data=df_trump).fit()
print("Trump")
print(model_private.summary2())

ALL
                            Results: Ordinary least squares
Model:                     OLS                       Adj. R-squared:          0.022    
Dependent Variable:        share_likely_private      AIC:                     2265.1433
Date:                      2025-11-14 06:38          BIC:                     2277.6206
No. Observations:          473                       Log-Likelihood:          -1129.6  
Df Model:                  2                         F-statistic:             6.386    
Df Residuals:              470                       Prob (F-statistic):      0.00183  
R-squared:                 0.026                     Scale:                   6.9914   
---------------------------------------------------------------------------------------
                                         Coef.  Std.Err.    t    P>|t|   [0.025  0.975]
---------------------------------------------------------------------------------------
Intercept                                3.1962   0.2104

# Conclusion: Republicans would share it either way\

In [ ]:
model_private = smf.ols('share_likely_public ~ C(condition_full)', data=df).fit()
print("ALL")
print(model_private.summary2())

model_private = smf.ols('share_likely_public ~ C(condition_full) * C(party, )', data=df_harris).fit()
print("Harris")

print(model_private.summary2())

model_private = smf.ols('share_likely_public ~ C(condition_full) * C(party, Treatment(reference="Republicans"))', data=df_trump).fit()
print("Trump")
print(model_private.summary2())

ALL
                            Results: Ordinary least squares
Model:                    OLS                       Adj. R-squared:           0.015    
Dependent Variable:       share_likely_public       AIC:                      2248.1278
Date:                     2025-11-14 06:38          BIC:                      2260.6051
No. Observations:         473                       Log-Likelihood:           -1121.1  
Df Model:                 2                         F-statistic:              4.713    
Df Residuals:             470                       Prob (F-statistic):       0.00941  
R-squared:                0.020                     Scale:                    6.7444   
---------------------------------------------------------------------------------------
                                         Coef.  Std.Err.    t    P>|t|   [0.025  0.975]
---------------------------------------------------------------------------------------
Intercept                                3.0316   0.2066

# we need more data (e.g. recruite 100 for each?)

In [ ]:
model_private = smf.ols('report_misleading ~ C(condition_full)', data=df).fit()
print("ALL")
print(model_private.summary2())

model_private = smf.ols('report_misleading ~ C(condition_full) * C(party, )', data=df_harris).fit()
print("Harris")

print(model_private.summary2())

model_private = smf.ols('report_misleading ~ C(condition_full) * C(party, Treatment(reference="Republicans"))', data=df_trump).fit()
print("Trump")
print(model_private.summary2())

ALL
                          Results: Ordinary least squares
Model:                    OLS                     Adj. R-squared:           0.033   
Dependent Variable:       report_misleading       AIC:                      472.2482
Date:                     2025-11-14 06:38        BIC:                      484.7255
No. Observations:         473                     Log-Likelihood:           -233.12 
Df Model:                 2                       F-statistic:              9.166   
Df Residuals:             470                     Prob (F-statistic):       0.000124
R-squared:                0.038                   Scale:                    0.15790 
------------------------------------------------------------------------------------
                                        Coef.  Std.Err.   t    P>|t|   [0.025 0.975]
------------------------------------------------------------------------------------
Intercept                               0.1266   0.0316 4.0041 0.0001  0.0645 0.1887
C(c

# Own party people would not report their own content as fake. Democrats whould just report Trump and thus only in the fake/contradicting case

In [ ]:
model_private = smf.ols('downrank_agree ~ C(condition_full)', data=df).fit()
print("ALL")
print(model_private.summary2())

model_private = smf.ols('downrank_agree ~ C(condition_full) * C(party, )', data=df_harris).fit()
print("Harris")

print(model_private.summary2())

model_private = smf.ols('downrank_agree ~ C(condition_full) * C(party, Treatment(reference="Republicans"))', data=df_trump).fit()
print("Trump")
print(model_private.summary2())

ALL
                           Results: Ordinary least squares
Model:                     OLS                    Adj. R-squared:           0.010    
Dependent Variable:        downrank_agree         AIC:                      2436.6003
Date:                      2025-11-14 06:38       BIC:                      2449.0776
No. Observations:          473                    Log-Likelihood:           -1215.3  
Df Model:                  2                      F-statistic:              3.429    
Df Residuals:              470                    Prob (F-statistic):       0.0332   
R-squared:                 0.014                  Scale:                    10.046   
-------------------------------------------------------------------------------------
                                        Coef.  Std.Err.    t    P>|t|   [0.025 0.975]
-------------------------------------------------------------------------------------
Intercept                               6.2785   0.2522 24.8993 0.0000  5.783

# Republicans agree less in Downranking, but overall people agree to downrank in treatment II

In [ ]:
model_private = smf.ols('watermark_action ~ C(condition_full)', data=df).fit()
print("ALL")
print(model_private.summary2())

model_private = smf.ols('watermark_action ~ C(condition_full) * C(party, )', data=df_harris).fit()
print("Harris")

print(model_private.summary2())

model_private = smf.ols('watermark_action ~ C(condition_full) * C(party, Treatment(reference="Republicans"))', data=df_trump).fit()
print("Trump")
print(model_private.summary2())

ALL
                           Results: Ordinary least squares
Model:                      OLS                    Adj. R-squared:           -0.003   
Dependent Variable:         watermark_action       AIC:                      2389.1495
Date:                       2025-11-14 06:38       BIC:                      2401.6268
No. Observations:           473                    Log-Likelihood:           -1191.6  
Df Model:                   2                      F-statistic:              0.3179   
Df Residuals:               470                    Prob (F-statistic):       0.728    
R-squared:                  0.001                  Scale:                    9.0871   
--------------------------------------------------------------------------------------
                                         Coef.  Std.Err.    t    P>|t|   [0.025 0.975]
--------------------------------------------------------------------------------------
Intercept                                5.0127   0.2398 20.9018 0.

# Deepfake perception does not change how they see watermarking. Maybe a bit when they are different party in treatment II and opposite candidate

In [ ]:
import numpy as np
from scipy.stats import norm

# Ensure the data is loaded and preprocessed (Section 1) before running this section.

# Determine actual condition (signal present = AI clip, signal absent = real clip)
df['isAI'] = df['group'].apply(lambda g: 1 if g != 1 else 0)  # 1 for AI-generated clip (groups 2 or 3), 0 for real (group 1)

# realness_perception is binary: 1 if participant perceived the clip as real, 0 if perceived as fake.
# We define: hit = perceived fake when clip is actually AI; false alarm = perceived fake when clip is actually real.

# Calculate hit rate, false alarm rate, and d' for each party
parties = ['Republicans', 'Democrats']
for party in parties:
    subset = df[df['party'] == party]
    # Proportions:
    hit_rate = np.mean(subset[subset['isAI'] == 1]['realness_perception'] == 0)         # AI clips identified as fake
    false_alarm_rate = np.mean(subset[subset['isAI'] == 0]['realness_perception'] == 0)  # Real clips mistakenly flagged as fake

    # Adjust extreme values (0 or 1) to avoid infinite z-scores
    eps = 1e-5
    if hit_rate == 1: hit_rate = 1 - eps
    if hit_rate == 0: hit_rate = eps
    if false_alarm_rate == 1: false_alarm_rate = 1 - eps
    if false_alarm_rate == 0: false_alarm_rate = eps

    # Compute d-prime = Z(hit_rate) - Z(false_alarm_rate)
    d_prime = norm.ppf(hit_rate) - norm.ppf(false_alarm_rate)
    print(f"\n{party}s:")
    print(f"  Hit rate = {hit_rate:.3f}   (proportion of AI clips correctly identified as fake)")
    print(f"  False alarm rate = {false_alarm_rate:.3f}   (proportion of real clips incorrectly flagged)")
    print(f"  d-prime = {d_prime:.3f}")


Republicanss:
  Hit rate = 0.406   (proportion of AI clips correctly identified as fake)
  False alarm rate = 0.197   (proportion of real clips incorrectly flagged)
  d-prime = 0.614

Democratss:
  Hit rate = 0.566   (proportion of AI clips correctly identified as fake)
  False alarm rate = 0.253   (proportion of real clips incorrectly flagged)
  d-prime = 0.832


In [ ]:
from scipy.stats import norm
from statsmodels.stats.proportion import proportions_ztest

df["isAI"] = df["group"].apply(lambda g: 1 if g in [2, 3] else 0)

speakers = df["speaker"].unique()
parties = ["Democrats", "Republicans"]
groups = [1, 2, 3]

# -------------------------------------------------------------------
# 2. Compute d′ BY SPEAKER × PARTY × GROUP
# -------------------------------------------------------------------
results = []

for speaker in speakers:
    for party in parties:
        for g in groups:

            sub = df[(df["speaker"] == speaker) &
                     (df["political_party"] == party) &
                     (df["group"] == g)]

            if len(sub) == 0:
                continue

            # Determine hits & false alarms WITHIN each group:
            # Hit = calling AI fake
            # FA = calling real fake

            if g == 1:
                # group 1 = REAL only → no hits
                hit_rate = np.nan
                # false alarms only
                fa_rate = np.mean(sub["realness_perception"] == 0)

            else:
                # group 2 or 3 = AI only → no false alarms
                hit_rate = np.mean(sub["realness_perception"] == 0)
                fa_rate = np.nan

            # For d':
            # Group 1: no hits → cannot compute full d'
            # Group 2/3: no false alarms → same issue
            # Instead compute d' only using full dataset (below)

            results.append([speaker, party, g, hit_rate, fa_rate])

df_group_stats = pd.DataFrame(
    results, columns=["speaker", "party", "group", "hit_rate", "false_alarm_rate"]
)

print("\n=== Hit & False-Alarm Rates by Speaker × Party × Group ===")
print(df_group_stats)

# -------------------------------------------------------------------
# 3. Compute d′ for GROUP 2 vs GROUP 3 ONLY
# (because only these have AI clips)
# -------------------------------------------------------------------
dprime_results = []

for speaker in speakers:
    for party in parties:
        # get ALL AI trials for this speaker × party
        sub_ai = df[(df["speaker"] == speaker) &
                    (df["political_party"] == party) &
                    (df["isAI"] == 1)]

        sub_real = df[(df["speaker"] == speaker) &
                      (df["political_party"] == party) &
                      (df["isAI"] == 0)]

        hit_rate = np.mean(sub_ai["realness_perception"] == 0)
        fa_rate = np.mean(sub_real["realness_perception"] == 0)

        # adjust extremes
        eps = 1e-5
        hit_rate = min(max(hit_rate, eps), 1 - eps)
        fa_rate = min(max(fa_rate, eps), 1 - eps)

        dprime = norm.ppf(hit_rate) - norm.ppf(fa_rate)

        dprime_results.append([speaker, party, hit_rate, fa_rate, dprime])

df_dprime = pd.DataFrame(
    dprime_results, columns=["speaker", "party", "hit_overall", "fa_overall", "dprime"]
)

print("\n=== d′ using ALL AI trials (Group 2 + 3) and ALL Real Trials ===")
print(df_dprime)


# -------------------------------------------------------------------
# 4. Z-tests for Dems vs Reps WITHIN each GROUP
# -------------------------------------------------------------------
test_results = []

for speaker in speakers:
    for g in groups:

        dsub = df[(df["speaker"] == speaker) &
                  (df["political_party"] == "Democrats") &
                  (df["group"] == g)]

        rsub = df[(df["speaker"] == speaker) &
                  (df["political_party"] == "Republicans") &
                  (df["group"] == g)]

        # HIT-RATE comparison for Group 2 and 3
        if g in [2, 3]:
            d_success = np.sum(dsub["realness_perception"] == 0)
            d_total = len(dsub)

            r_success = np.sum(rsub["realness_perception"] == 0)
            r_total = len(rsub)

            if d_total > 0 and r_total > 0:
                z, p = proportions_ztest([d_success, r_success],
                                         [d_total, r_total])
                test_results.append([speaker, g, "hit_rate", z, p])

        # FALSE ALARMS for Group 1 only
        if g == 1:
            d_success = np.sum(dsub["realness_perception"] == 0)
            d_total = len(dsub)

            r_success = np.sum(rsub["realness_perception"] == 0)
            r_total = len(rsub)

            if d_total > 0 and r_total > 0:
                z, p = proportions_ztest([d_success, r_success],
                                         [d_total, r_total])
                test_results.append([speaker, g, "false_alarm", z, p])

df_ztests = pd.DataFrame(
    test_results, columns=["speaker", "group", "metric", "z", "p"]
)

print("\n=== Z-tests for Group-Specific Comparisons (Dem vs Rep) ===")
print(df_ztests)


=== Hit & False-Alarm Rates by Speaker × Party × Group ===
   speaker        party  group  hit_rate  false_alarm_rate
0    Trump    Democrats      1       NaN          0.325581
1    Trump    Democrats      2  0.744681               NaN
2    Trump    Democrats      3  0.829787               NaN
3    Trump  Republicans      1       NaN          0.200000
4    Trump  Republicans      2  0.350000               NaN
5    Trump  Republicans      3  0.480000               NaN
6   Harris    Democrats      1       NaN          0.181818
7   Harris    Democrats      2  0.170732               NaN
8   Harris    Democrats      3  0.468085               NaN
9   Harris  Republicans      1       NaN          0.192308
10  Harris  Republicans      2  0.300000               NaN
11  Harris  Republicans      3  0.500000               NaN

=== d′ using ALL AI trials (Group 2 + 3) and ALL Real Trials ===
  speaker        party  hit_overall  fa_overall    dprime
0   Trump    Democrats     0.787234    0.325581  

# Democrats are better in understanding that what trump says is fake. There is the same tendency for reps and harris, but the difference is not statistically significant

In [ ]:
df["fell_for_deepfake"] = (
    (df["group"] != 1) &
    (df["realness_perception"] == 1)
).astype(int)

# Also create a clean party variable if needed
df["party"] = df["political_party"]  # shorthand

df_fell_for_deepfake = df[df["fell_for_deepfake"]==1]
df_fell_for_deepfake_harris = df_fell_for_deepfake[df_fell_for_deepfake['speaker']=="Harris"]
df_fell_for_deepfake_trump = df_fell_for_deepfake[df_fell_for_deepfake['speaker']=="Trump"]


df_resist_deepfake = df[df["fell_for_deepfake"]!=1]

# -----------------------------------------------------------
# 2. MODEL: ALL respondents
# -----------------------------------------------------------
print("\n\n======================")
print("MODEL 1: ALL respondents")
print("======================")

model_all = smf.ols(
    'downrank_agree ~ C(condition_full)',
    data=df_fell_for_deepfake
).fit()

print(model_all.summary2())

model_all = smf.ols(
    'share_likely_private ~ C(condition_full)',
    data=df_fell_for_deepfake
).fit()

print(model_all.summary2())

model_all = smf.ols(
    'share_likely_public ~ C(condition_full)',
    data=df_fell_for_deepfake
).fit()


model_all = smf.ols(
    'report_misleading ~ C(condition_full)',
    data=df_fell_for_deepfake
).fit()

model_all = smf.ols(
    'watermark_action ~ C(condition_full)',
    data=df_fell_for_deepfake
).fit()

print(model_all.summary2())

# -----------------------------------------------------------
# 3. MODEL: HARRIS only
# -----------------------------------------------------------
df_harris = df[df["speaker"] == "Harris"].copy()

print("\n\n======================")
print("MODEL 2: HARRIS only")
print("======================")

model_harris = smf.ols(
    'downrank_agree ~ C(condition_full) *  C(party)',
    data=df_fell_for_deepfake_harris
).fit()

print(model_harris.summary2())


# -----------------------------------------------------------
# 4. MODEL: TRUMP only (Republicans = reference)
# -----------------------------------------------------------
df_trump = df[df["speaker"] == "Trump"].copy()

print("\n\n======================")
print("MODEL 3: TRUMP only (Republicans = reference)")
print("======================")

model_trump = smf.ols(
    'downrank_agree ~ C(condition_full) * C(party, Treatment(reference="Republicans"))',
    data=df_fell_for_deepfake_trump
).fit()

print(model_trump.summary2())




MODEL 1: ALL respondents
                           Results: Ordinary least squares
Model:                     OLS                     Adj. R-squared:            -0.001  
Dependent Variable:        downrank_agree          AIC:                       824.8310
Date:                      2025-11-14 06:38        BIC:                       830.9561
No. Observations:          158                     Log-Likelihood:            -410.42 
Df Model:                  1                       F-statistic:               0.8118  
Df Residuals:              156                     Prob (F-statistic):        0.369   
R-squared:                 0.005                   Scale:                     10.697  
--------------------------------------------------------------------------------------
                                         Coef.  Std.Err.    t    P>|t|   [0.025 0.975]
--------------------------------------------------------------------------------------
Intercept                                6.3

In [ ]:
from scipy.stats import ttest_ind
df_resist = df[df["fell_for_deepfake"]!=1]
df_fell = df_fell_for_deepfake
# Outcomes to compare
vars_to_compare = [
    "policy_agreement",
    "likelihood_to_vote",
    "share_likely_private",
    "share_likely_public",
    "report_misleading",
    "downrank_agree",
    "watermark_action",
    "stance_before",
    "stance_after",
    "salience_before",
    "salience_after"
]

print("\n===== MEAN COMPARISONS: RESISTORS vs FALLERS =====\n")

results = []

for var in vars_to_compare:
    x = df_resist[var].dropna()
    y = df_fell[var].dropna()

    # Welch t-test
    t, p = ttest_ind(x, y, equal_var=False)

    results.append([
        var,
        x.mean(),
        y.mean(),
        t,
        p
    ])

# Convert to table
res_df = pd.DataFrame(
    results,
    columns=["variable", "mean_resist", "mean_fell", "t_value", "p_value"]
)

print(res_df.to_string(index=False))


===== MEAN COMPARISONS: RESISTORS vs FALLERS =====

            variable  mean_resist  mean_fell   t_value      p_value
    policy_agreement     5.600000   6.550633 -3.046812 2.493689e-03
  likelihood_to_vote     4.739683   6.303797 -4.452901 1.156984e-05
share_likely_private     2.431746   2.943038 -1.916258 5.630388e-02
 share_likely_public     2.326984   2.968354 -2.425693 1.590645e-02
   report_misleading     0.282540   0.050633  7.517129 2.850678e-13
      downrank_agree     7.133333   6.126582  3.215640 1.443803e-03
    watermark_action     4.847619   5.018987 -0.612535 5.405728e-01
       stance_before     6.330159   6.493671 -0.559948 5.759016e-01
        stance_after     6.387302   6.506329 -0.409216 6.826377e-01
     salience_before     7.380952   7.354430  0.104608 9.167499e-01
      salience_after     7.425397   7.227848  0.762048 4.465598e-01


In [ ]:
model_policy = smf.ols('policy_agreement ~ C(condition_full)*C(party) + C(party)*C(speaker) + C(condition_full)*C(speaker) + C(topic)', data=df).fit()
print("\nOLS Regression: Policy Agreement ~ Condition, Party, Speaker, Topic (+ interactions)")
print(model_policy.summary2())

model_vote = smf.ols('likelihood_to_vote ~ C(condition_full)*C(party) + C(party)*C(speaker) + C(condition_full)*C(speaker) + C(topic)', data=df).fit()
print("\nOLS Regression: Vote Intention ~ Condition, Party, Speaker, Topic (+ interactions)")
print(model_vote.summary2())



OLS Regression: Policy Agreement ~ Condition, Party, Speaker, Topic (+ interactions)
                                        Results: Ordinary least squares
Model:                             OLS                             Adj. R-squared:                    0.209    
Dependent Variable:                policy_agreement                AIC:                               2382.6844
Date:                              2025-11-14 06:38                BIC:                               2449.2299
No. Observations:                  473                             Log-Likelihood:                    -1175.3  
Df Model:                          15                              F-statistic:                       9.338    
Df Residuals:                      457                             Prob (F-statistic):                3.35e-19 
R-squared:                         0.235                           Scale:                             8.7257   
----------------------------------------------------------

In [ ]:
model_trust = smf.ols('speaker_trustworthiness ~ C(condition_full)*C(party) + C(party)*C(speaker) + C(condition_full)*C(speaker) + C(topic)', data=df).fit()
print("\nOLS Regression: Trust in Speaker ~ Condition, Party, Speaker, Topic (+ interactions)")
print(model_trust.summary2())


OLS Regression: Trust in Speaker ~ Condition, Party, Speaker, Topic (+ interactions)
                                        Results: Ordinary least squares
Model:                            OLS                                 Adj. R-squared:                 0.359    
Dependent Variable:               speaker_trustworthiness             AIC:                            2282.3549
Date:                             2025-11-14 06:38                    BIC:                            2348.9004
No. Observations:                 473                                 Log-Likelihood:                 -1125.2  
Df Model:                         15                                  F-statistic:                    18.66    
Df Residuals:                     457                                 Prob (F-statistic):             9.61e-39 
R-squared:                        0.380                               Scale:                          7.0580   
----------------------------------------------------------

In [ ]:
emotions = ['em_anger', 'em_fear', 'em_enthusiasm', 'em_pride']
for emo in emotions:
    model_emo = smf.ols(f'{emo} ~ C(condition_full)*C(party) + C(party)*C(speaker) + C(condition_full)*C(speaker) + C(topic)', data=df).fit()
    print(f"\nOLS Regression: Emotional Reaction ({emo}) ~ Condition, Party, Speaker, Topic (+ interactions)")
    print(model_emo.summary2())


OLS Regression: Emotional Reaction (em_anger) ~ Condition, Party, Speaker, Topic (+ interactions)
                                        Results: Ordinary least squares
Model:                             OLS                             Adj. R-squared:                    0.115    
Dependent Variable:                em_anger                        AIC:                               2366.8049
Date:                              2025-11-14 06:38                BIC:                               2433.3504
No. Observations:                  473                             Log-Likelihood:                    -1167.4  
Df Model:                          15                              F-statistic:                       5.095    
Df Residuals:                      457                             Prob (F-statistic):                2.26e-09 
R-squared:                         0.143                           Scale:                             8.4376   
---------------------------------------------

In [ ]:
df_all = df
df_all['condition'] = df_all['group'].map({1: 'Real', 2: 'AI', 3: 'AI'})  # binary condition: Real vs any AI

# Fill NaNs in MIP topic lists with empty string for parsing
df_all['mip_topics_before'] = df_all['mip_topics_before'].fillna('')
df_all['mip_topics'] = df_all['mip_topics'].fillna('')

# Function to parse a comma-separated string of topics into a normalized set of topic keywords
def parse_topics(topic_str):
    if topic_str.strip() == '':
        return set()
    # Split by commas and strip spaces
    topics = [t.strip() for t in topic_str.split(',') if t.strip()]
    # Normalize by removing spaces and special characters for comparison
    topics_norm = set(t.lower().replace(' ', '').replace('/', '').replace('&', '') for t in topics)
    return topics_norm

# Determine added and removed status for each participant's clip topic
added_flags = []
removed_flags = []

for _, row in df_all.iterrows():
    before_set = parse_topics(row['mip_topics_before'])
    after_set = parse_topics(row['mip_topics'])
    # Normalize the clip topic for comparison (remove spaces, etc.)
    clip_topic_norm = str(row['topic']).lower().replace(' ', '').replace('/', '').replace('&', '')
    # Added if clip's topic was not in before_set but appears in after_set
    added = 1 if (clip_topic_norm not in before_set and clip_topic_norm in after_set) else 0
    # Removed if clip's topic was in before_set but no longer in after_set
    removed = 1 if (clip_topic_norm in before_set and clip_topic_norm not in after_set) else 0
    added_flags.append(added)
    removed_flags.append(removed)

df_all['added_topic'] = added_flags
df_all['removed_topic'] = removed_flags

# Show how many participants added or removed the clip's issue, by issue topic
added_counts = df_all.groupby('topic')['added_topic'].sum()
removed_counts = df_all.groupby('topic')['removed_topic'].sum()
mip_change = pd.DataFrame({'AddedCount': added_counts, 'RemovedCount': removed_counts})
print("Number of participants who added or removed each issue as a MIP:")
print(mip_change)

# Logistic regression: Does adding the clip's topic to MIP depend on condition (real vs AI) and the issue topic?
logit_added = smf.logit('added_topic ~ C(condition) + C(party) + C(speaker)', data=df_all).fit(disp=False)
print("\nLogistic Regression: P(added clip topic to MIP) ~ Condition (Real vs AI) + Topic")
print(logit_added.summary2())

mentioned_after_flags = []

for _, row in df_all.iterrows():
    before_set = parse_topics(row['mip_topics_before'])
    after_set = parse_topics(row['mip_topics'])

    # Normalize topic text
    clip_topic_norm = str(row['topic']).lower().replace(' ', '').replace('/', '').replace('&', '')

    # NEW: probability variable → 1 if mentioned after, else 0
    mentioned_after = 1 if clip_topic_norm in after_set else 0

    mentioned_after_flags.append(mentioned_after)

df_all['mip_clip_topic_after'] = mentioned_after_flags

Number of participants who added or removed each issue as a MIP:
                  AddedCount  RemovedCount
topic                                     
Climate                    5             2
Economy                    5             4
Education                  8             1
Gender                     3             1
Immigration                4             2
NationalSecurity          11             1
Racism                     5             3

Logistic Regression: P(added clip topic to MIP) ~ Condition (Real vs AI) + Topic
                            Results: Logit
Model:                  Logit              Method:             MLE     
Dependent Variable:     added_topic        Pseudo R-squared:   0.002   
Date:                   2025-11-14 06:38   AIC:                286.2045
No. Observations:       473                BIC:                302.8409
Df Model:               3                  Log-Likelihood:     -139.10 
Df Residuals:           469                LL-Null:            

In [ ]:
df_all.groupby("group")['mip_clip_topic_after'].mean()
df_all.groupby("group")['mip_clip_topic_after'].std()

means = df.groupby('group')['mip_clip_topic_after'].mean()

groups = [1,2,3]
tests=[]
for i in range(len(groups)):
    for j in range(i+1,len(groups)):
        g1=groups[i]; g2=groups[j]
        x = df[df.group==g1]['mip_clip_topic_after']
        y = df[df.group==g2]['mip_clip_topic_after']
        t,p = ttest_ind(x,y,equal_var=False)
        tests.append([f"{g1} vs {g2}", t, p])

print(means)
print(df_all.groupby("group")['mip_clip_topic_after'].std())
print(pd.DataFrame(tests, columns=['comparison','t','p']))

group
1    0.424051
2    0.341772
3    0.356688
Name: mip_clip_topic_after, dtype: float64
group
1    0.495769
2    0.475812
3    0.480554
Name: mip_clip_topic_after, dtype: float64
  comparison         t         p
0     1 vs 2  1.505078  0.133311
1     1 vs 3  1.224469  0.221697
2     2 vs 3 -0.276797  0.782119


In [ ]:
df_all.groupby("speaker")['mip_clip_topic_after'].mean()

,mip_clip_topic_after
speaker,
Harris,0.367257
Trump,0.380567


In [ ]:
df_all.groupby(["speaker", "party"])['mip_clip_topic_after'].mean()

speaker  party      
Harris   Democrats      0.378788
         Republicans    0.351064
Trump    Democrats      0.357664
         Republicans    0.409091
Name: mip_clip_topic_after, dtype: float64

In [ ]:
import statsmodels.formula.api as smf

model = smf.logit("mip_clip_topic_after ~ C(group) + C(speaker) + C(political_party)", data=df_all).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.658288
         Iterations 4
                            Logit Regression Results                            
Dep. Variable:     mip_clip_topic_after   No. Observations:                  473
Model:                            Logit   Df Residuals:                      468
Method:                             MLE   Df Model:                            4
Date:                  Fri, 14 Nov 2025   Pseudo R-squ.:                0.004339
Time:                          06:38:29   Log-Likelihood:                -311.37
converged:                         True   LL-Null:                       -312.73
Covariance Type:              nonrobust   LLR p-value:                    0.6068
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            -0.3557  

In [ ]:
model = smf.logit("mip_clip_topic_after ~ C(group) + C(speaker)*C(political_party)", data=df_all).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.657655
         Iterations 4
                            Logit Regression Results                            
Dep. Variable:     mip_clip_topic_after   No. Observations:                  473
Model:                            Logit   Df Residuals:                      467
Method:                             MLE   Df Model:                            5
Date:                  Fri, 14 Nov 2025   Pseudo R-squ.:                0.005297
Time:                          06:38:29   Log-Likelihood:                -311.07
converged:                         True   LL-Null:                       -312.73
Covariance Type:              nonrobust   LLR p-value:                    0.6518
                                                            coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------------------
Interc